In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight

## Anaerobic Exercise Protocol

### First version (S01 to S18) - Males

| Phase | Duration | Label |
|-------|----------|-------|
| Warm up | ~0:35 | REST |
| Sprint 1 | ~4:00 | EXERCISE |
| Recovery 1 | ~0:30 | REST |
| Sprint 2 | ~3:30 | EXERCISE |
| Recovery 2 | ~0:30 | REST |
| Sprint 3 | ~4:00 | EXERCISE |
| Cool Down | ~4:00 | REST |

**Total tags expected: 7** (transitions between phases)

### Second version (f01 to f13) - Females

| Phase | Duration | Label |
|-------|----------|-------|
| Baseline | ~3:00 | REST |
| Warm up | ~0:40 | REST |
| Sprint 1 | ~3:30 | EXERCISE |
| Recovery 1 | ~0:35 | REST |
| Sprint 2 | ~3:20 | EXERCISE |
| Recovery 2 | ~0:30 | REST |
| Sprint 3 | ~3:00 | EXERCISE |
| Recovery 3 | ~0:30 | REST |
| Sprint 4 | ~3:15 | EXERCISE |
| Cool Down | ~2:30 | REST |
| Rest | ~2:00 | REST |

**Total tags expected: 11** (transitions between phases)

**Key Differences from V1:**
- V2 includes a Baseline period (not in V1)
- V2 has 4 sprint intervals; V1 has 3 sprint intervals
- V2 has more recovery periods between sprints
- V1 has longer sprint durations

# Anaerobic Exercise Detection from Wearable Physiological Signals

**Goal**: Build a classifier to detect anaerobic exercise (high-intensity sprint cycling) vs rest states using physiological data from wearable devices.

Build an "Anaerobic Exercise Detector"

1. **Collect the Signals**  
   Wearable sensors data (Empatica E4) to gather raw physiological data during high-intensity sprint protocol.

2. **Label EXERCISE vs REST Periods**  
   Mark time segments where the participant was performing all-out sprints or recovering/resting.

3. **Calculate Biomarkers**
   These features help the model understand what's happening in the body:
   - **HRV (Heart Rate Variability)**  
     *How much heartbeat timing changes between beats.*  
     → **Lower during sprints** (heart becomes more "robot-like")
   - **EDA (Electrodermal Activity)**  
     *Measures sweat gland activity.*  
     → **Higher during intense exercise**
   - **HR (Heart Rate)**  
     *How fast the heart is beating.*  
     → **Significantly higher during sprints**
   - **ACC (Accelerometer / Movement)**  
     *Body movement from explosive cycling.*  
     → **Very high and intense during sprints**


## Dataset Overview:
- **22 subjects** (10 males V1 protocol, 12 females V2 protocol)
- **Signals**: EDA, BVP, HR, IBI (for HRV), Temperature, Accelerometer
- **Exercise Protocol**: High-intensity sprint intervals with recovery periods
- **Rest Periods**: Warm up, Cool Down, Recovery, Baseline, Rest

In [ ]:
dataset_path = '22subjects/ANAEROBIC'
subject_info_path = 'WISE_data_files/subject-info.csv'

In [ ]:
def create_df_array(dataframe):
    """Converts a pandas DataFrame to a flattened numpy array."""
    return dataframe.values.flatten()


def time_abs_(UTC_array):
    """Converts UTC timestamps to seconds from the start of recording."""
    new_array = []
    start_time = datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')
    
    for utc in UTC_array:
        current_time = datetime.datetime.strptime(utc, '%Y-%m-%d %H:%M:%S')
        seconds_elapsed = (current_time - start_time).total_seconds()
        new_array.append(int(seconds_elapsed))
    
    return new_array


def moving_average(acc_data):
    """
    Applies a moving average filter to accelerometer data to measure movement.
    Higher values = more movement, Lower values = less movement
    """
    avg = 0
    prevX, prevY, prevZ = 0, 0, 0
    results = []
    
    # Process every second (32 samples at 32 Hz)
    for i in range(0, len(acc_data), 32):
        sum_ = 0
        buffX = acc_data[i:i+32, 0]
        buffY = acc_data[i:i+32, 1]
        buffZ = acc_data[i:i+32, 2]
        
        for j in range(len(buffX)):
            sum_ += max(
                abs(buffX[j] - prevX),
                abs(buffY[j] - prevY),
                abs(buffZ[j] - prevZ)
            )
            prevX, prevY, prevZ = buffX[j], buffY[j], buffZ[j]
        
        avg = avg * 0.9 + (sum_ / 32) * 0.1
        results.append(avg)
    
    return results

print("Helper functions defined")

In [ ]:
def read_signals(main_folder):
    """
    Each subject folder contains: EDA, BVP, HR, IBI, TEMP, ACC, tags
    """
    signal_dict = {}
    time_dict = {}
    fs_dict = {}
    
    subfolders = next(os.walk(main_folder))[1]
    
    # Get start times
    utc_start_dict = {}
    for folder_name in subfolders:
        csv_path = f'{main_folder}/{folder_name}/EDA.csv'
        df = pd.read_csv(csv_path)
        utc_start_dict[folder_name] = df.columns.tolist()
    
    # Read all signals
    for folder_name in subfolders:
        folder_path = os.path.join(main_folder, folder_name)
        files = os.listdir(folder_path)
        
        signals = {}
        time_line = {}
        fs_signal = {}
        
        desired_files = ['EDA.csv', 'BVP.csv', 'HR.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv', 'IBI.csv']
        
        for file_name in files:
            if file_name not in desired_files:
                continue
            
            file_path = os.path.join(folder_path, file_name)
            signal_name = file_name.replace('.csv', '')
            
            if file_name == 'tags.csv':
                try:
                    df = pd.read_csv(file_path, header=None)
                    tags_vector = create_df_array(df)
                    tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                    signal_array = time_abs_(tags_UTC_vector)
                except pd.errors.EmptyDataError:
                    signal_array = []
            
            elif file_name == 'IBI.csv':
                df = pd.read_csv(file_path)
                signal_array = df.values
                fs_signal['IBI'] = 'variable'
            
            else:
                df = pd.read_csv(file_path)
                fs = int(df.iloc[0, 0])
                signal_array = df.iloc[1:].values
                time_array = np.linspace(0, len(signal_array)/fs, len(signal_array))
                
                time_line[signal_name] = time_array
                fs_signal[signal_name] = fs
            
            signals[signal_name] = signal_array
        
        signal_dict[folder_name] = signals
        time_dict[folder_name] = time_line
        fs_dict[folder_name] = fs_signal
    
    return signal_dict, time_dict, fs_dict

print(" Data loading function defined")

In [ ]:
# Load all physiological signals
print("Loading physiological signals...")
signal_data, time_data, fs_dict = read_signals(dataset_path)

subjects = list(signal_data.keys())
v1_subjects = sorted([s for s in subjects if s.startswith('S')])
v2_subjects = sorted([s for s in subjects if s.startswith('f')])

print(f"\n✓ Loaded {len(subjects)} subjects:")
print(f"   V1: {v1_subjects}")
print(f"   V2: {v2_subjects}")

In [ ]:
# Creates time segments labeled as EXERCISE or REST based on Anaerobic protocol tags.
def get_anaerobic_rest_segments(subject_id, tags):
    """
    Segment the Anaerobic protocol into EXERCISE vs REST periods.

    V1 (S01-S18): Warm up -> Sprint 1 -> Recovery 1 -> Sprint 2 -> Recovery 2 -> Sprint 3 -> Cool Down
    V2 (f01-f13): Baseline -> Warm up -> Sprint 1-4 with Recoveries -> Cool Down -> Rest

    EXERCISE: High-intensity sprint phases (maximal effort)
    REST: Warm up, Cool Down, Recovery, Baseline, Rest (low intensity/recovery)
    """
    segments = []
    
    if subject_id.startswith('S'):  # V1 protocol (S01-S18)
        if len(tags) >= 7:
            # Warm up: REST
            segments.append({'start': tags[0], 'end': tags[1], 'label': 'REST', 'phase': 'Warm up'})
            
            # Sprint 1: EXERCISE
            segments.append({'start': tags[1], 'end': tags[2], 'label': 'EXERCISE', 'phase': 'Sprint 1'})
            
            # Recovery 1: REST
            segments.append({'start': tags[2], 'end': tags[3], 'label': 'REST', 'phase': 'Recovery 1'})
            
            # Sprint 2: EXERCISE
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'EXERCISE', 'phase': 'Sprint 2'})
            
            # Recovery 2: REST
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'REST', 'phase': 'Recovery 2'})
            
            # Sprint 3: EXERCISE
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'EXERCISE', 'phase': 'Sprint 3'})
            
            # Cool Down: REST (if tag exists)
            if len(tags) >= 8:
                segments.append({'start': tags[6], 'end': tags[7], 'label': 'REST', 'phase': 'Cool Down'})
    
    else:  # V2 protocol (f01-f13)
        if len(tags) >= 11:
            # Baseline: REST
            segments.append({'start': tags[0], 'end': tags[1], 'label': 'REST', 'phase': 'Baseline'})
            
            # Warm up: REST
            segments.append({'start': tags[1], 'end': tags[2], 'label': 'REST', 'phase': 'Warm up'})
            
            # Sprint 1: EXERCISE
            segments.append({'start': tags[2], 'end': tags[3], 'label': 'EXERCISE', 'phase': 'Sprint 1'})
            
            # Recovery 1: REST
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'REST', 'phase': 'Recovery 1'})
            
            # Sprint 2: EXERCISE
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'EXERCISE', 'phase': 'Sprint 2'})
            
            # Recovery 2: REST
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'REST', 'phase': 'Recovery 2'})
            
            # Sprint 3: EXERCISE
            segments.append({'start': tags[6], 'end': tags[7], 'label': 'EXERCISE', 'phase': 'Sprint 3'})
            
            # Recovery 3: REST
            segments.append({'start': tags[7], 'end': tags[8], 'label': 'REST', 'phase': 'Recovery 3'})
            
            # Sprint 4: EXERCISE
            segments.append({'start': tags[8], 'end': tags[9], 'label': 'EXERCISE', 'phase': 'Sprint 4'})
            
            # Cool Down: REST
            segments.append({'start': tags[9], 'end': tags[10], 'label': 'REST', 'phase': 'Cool Down'})
            
            # Rest: REST (if tag exists)
            if len(tags) >= 12:
                segments.append({'start': tags[10], 'end': tags[11], 'label': 'REST', 'phase': 'Rest'})
    
    return segments

print("Segmentation function defined")

---
### Segmentation

`get_anaerobic_rest_segments(subject_id, tags)`
- Divides protocol into EXERCISE vs REST periods
- Uses tag timestamps to identify phase transitions
- V1 has 7 tags (6 transitions), V2 has 11 tags (10 transitions)

In [ ]:
# Create segments for all subjects
all_segments = {}
for subject in subjects:
    tags = signal_data[subject]['tags']
    if len(tags) > 0:
        segments = get_anaerobic_rest_segments(subject, tags)
        all_segments[subject] = segments

# Show example for one V1 and one V2 subject
print(f"Example V1 subject - {v1_subjects[0]}:")
print(f"  Tags available: {len(signal_data[v1_subjects[0]]['tags'])}")
for seg in all_segments.get(v1_subjects[0], []):
    dur = seg['end'] - seg['start']
    print(f"  {seg['label']:8s} {seg['phase']:15s} {seg['start']:4d}-{seg['end']:4d}s ({dur}s)")

print(f"\nExample V2 subject - {v2_subjects[0]}:")
print(f"  Tags available: {len(signal_data[v2_subjects[0]]['tags'])}")
for seg in all_segments.get(v2_subjects[0], []):
    dur = seg['end'] - seg['start']
    print(f"  {seg['label']:8s} {seg['phase']:15s} {seg['start']:4d}-{seg['end']:4d}s ({dur}s)")

### V1 Tag Mapping (S01-S18)

| Tag Index | Phase Transition | Label |
|-----------|------------------|-------|
| 0 → 1 | Warm up | REST |
| 1 → 2 | Sprint 1 | EXERCISE |
| 2 → 3 | Recovery 1 | REST |
| 3 → 4 | Sprint 2 | EXERCISE |
| 4 → 5 | Recovery 2 | REST |
| 5 → 6 | Sprint 3 | EXERCISE |
| 6 → 7 | Cool Down (if exists) | REST |

### V2 Tag Mapping (f01-f13)

| Tag Index | Phase Transition | Label |
|-----------|------------------|-------|
| 0 → 1 | Baseline | REST |
| 1 → 2 | Warm up | REST |
| 2 → 3 | Sprint 1 | EXERCISE |
| 3 → 4 | Recovery 1 | REST |
| 4 → 5 | Sprint 2 | EXERCISE |
| 5 → 6 | Recovery 2 | REST |
| 6 → 7 | Sprint 3 | EXERCISE |
| 7 → 8 | Recovery 3 | REST |
| 8 → 9 | Sprint 4 | EXERCISE |
| 9 → 10 | Cool Down | REST |
| 10 → 11 | Rest (if exists) | REST |

---
### Visualize Raw Signals

In [ ]:
# Plotting function to visualize all physiological signals for one subject.
def plot_subject_signals(subject_id, signals, time_dict, tags, state='ANAEROBIC'):
    """
    Plot all physiological signals for one subject.
    Highlights EXERCISE periods with colored shading to verify segmentation.
    """
    plt.figure(figsize=(20, 12))
    
    signal_names = ['EDA', 'BVP', 'HR', 'TEMP', 'ACC']
    available_signals = [s for s in signal_names if s in signals and s in time_dict]
    
    for i, signal_name in enumerate(available_signals, 1):
        plt.subplot(len(available_signals), 1, i)
        
        if i == 1:
            plt.title(f'{subject_id} - {state} Protocol (Raw Signals)',
                     fontsize=16, fontweight='bold', pad=10)
        
        # Plot signal
        if signal_name == 'ACC':
            acc_filtered = moving_average(signals[signal_name])
            plt.plot(acc_filtered, label=signal_name, linewidth=1, color='purple')
        else:
            plt.plot(time_dict[signal_name], signals[signal_name],
                    label=signal_name, linewidth=1)
        
        # Add tag markers (red vertical lines)
        for j, tag in enumerate(tags[1:], 1):
            plt.axvline(x=tag, color='red', linestyle='--', alpha=0.5, linewidth=1)
        
        # Highlight EXERCISE periods with colored shading
        if state == 'ANAEROBIC' and len(tags) > 0:
            if subject_id.startswith('S'):  # V1 protocol
                if len(tags) >= 7:
                    # Sprint intervals
                    sprint_phases = [
                        (1, 2, 'red', 'Sprint 1'),
                        (3, 4, 'darkred', 'Sprint 2'),
                        (5, 6, 'maroon', 'Sprint 3')
                    ]
                    for start_idx, end_idx, color, label in sprint_phases:
                        if end_idx < len(tags):
                            plt.axvspan(tags[start_idx], tags[end_idx], 
                                       color=color, alpha=0.3,
                                       label=label if i == 1 else '')
            else:  # V2 protocol
                if len(tags) >= 11:
                    # Sprint intervals for V2
                    sprint_phases = [
                        (2, 3, 'orange', 'Sprint 1'),
                        (4, 5, 'orangered', 'Sprint 2'),
                        (6, 7, 'red', 'Sprint 3'),
                        (8, 9, 'darkred', 'Sprint 4')
                    ]
                    for start_idx, end_idx, color, label in sprint_phases:
                        if end_idx < len(tags):
                            plt.axvspan(tags[start_idx], tags[end_idx], 
                                       color=color, alpha=0.3,
                                       label=label if i == 1 else '')

        plt.ylabel(signal_name, fontsize=11, fontweight='bold')
        plt.grid(True, alpha=0.3)
        if i == 1:
            plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), 
                      fontsize=9, framealpha=0.95, borderaxespad=0)
    
    plt.xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Plotting function defined")

- **V1 (males):** S04, S05, S08, S09, S10, S13, S14, S15, S17, S18
- **V2 (females):** f01, f02, f03, f04, f05, f06, f08, f09, f10, f11, f12, f13

- Red dashed lines: Protocol transition markers (tags)
- Red/Orange shading: High-intensity sprint periods
- EDA (top): Look for intense sweat response during sprints
- HR (middle): Should show very elevated heart rate during sprint phases
- ACC (bottom): Explosive movement patterns from maximal effort cycling
- Goal: verify that sprint periods show extreme physiological changes

In [ ]:
# Specify subject id to visualize
subject_id = 'S04'  # Options: S04, S05, S08, S09, S10, S13, S14, S15, S17, S18, f01-f13

# Plot the selected subject
if subject_id in signal_data:
    print(f"Visualizing subject: {subject_id}")
    protocol = "V1 (males, 3 sprints)" if subject_id.startswith('S') else "V2 (females, 4 sprints)"
    print(f"Protocol: {protocol}\n")
    
    plot_subject_signals(
        subject_id,
        signal_data[subject_id],
        time_data[subject_id],
        signal_data[subject_id]['tags'],
        state='ANAEROBIC'
    )
else:
    print(f"Subject '{subject_id}' not found!")
    print(f"Available subjects: {subjects}")

In [ ]:
# PLOT FOR V2 Subject (female)
subject_f01 = 'f01'
protocol = "V1 (males, 3 sprints)" if subject_f01.startswith('S') else "V2 (females, 4 sprints)"
print(f"Protocol: {protocol}\n")

plot_subject_signals(
    subject_f01,
    signal_data[subject_f01],
    time_data[subject_f01],
    signal_data[subject_f01]['tags'],
    state='ANAEROBIC'
)

----

## Calculate Biomarkers (Features)

### Heart Rate Variability (HRV) Function

We need calculate two main HRV metrics from the IBI (inter-beat interval) signal:

##### 1. **SDNN** — Overall Heartbeat Variability  
**SDNN = standard deviation of the IBI values**
- Overall heartbeat variability across the entire segment
- Measures how much your heartbeat timing *changes in general*.
- Think of your heartbeat like footsteps:
  - **Resting:** footsteps naturally speed up and slow down → **high SDNN**
  - **Sprinting:** footsteps are evenly spaced like a metronome → **low SDNN**

##### 2. **RMSSD** — Beat-to-Beat Variability 
- Beat-to-beat variability (short-term changes)
- More sensitive to rapid changes than SDNN
**RMSSD = root mean square of the differences between consecutive IBIs**

HRV is one of the most reliable exercise intensity biomarkers
- During high-intensity exercise, the sympathetic nervous system dominates → reduced variability
- High HRV (REST): Parasympathetic dominance = healthy variability
- Low HRV (EXERCISE): Sympathetic dominance = rigid heart rate

In [ ]:
# calculate_hrv(ibi_data, start_time, end_time)
# - Calculates Heart Rate Variability metrics:
# - SDNN: Standard deviation of IBI intervals (overall HRV)
# - RMSSD: Root mean square of successive differences (short-term HRV)
# -> Lower HRV = Higher exercise intensity

def calculate_hrv(ibi_data, start_time, end_time):
    """Calculate HRV: Lower HRV = More Intense Exercise"""
    if len(ibi_data) == 0:
        return {'SDNN': np.nan, 'RMSSD': np.nan}
    # 1. Filter IBI data for the time segment
    timestamps = ibi_data[:, 0]
    ibi_values = ibi_data[:, 1]
    mask = (timestamps >= start_time) & (timestamps <= end_time)
    segment_ibi = ibi_values[mask]
    
    if len(segment_ibi) < 2:
        return {'SDNN': np.nan, 'RMSSD': np.nan}
    # 2. Calculate SDNN (Standard Deviation of NN intervals)
    sdnn = np.std(segment_ibi)
    # 3. Calculate RMSSD (Root Mean Square of Successive Differences)
    rmssd = np.sqrt(np.mean(np.diff(segment_ibi)**2))  # Short-term HRV
    
    return {'SDNN': sdnn, 'RMSSD': rmssd}

print("HRV calculation function defined")

### Extract Features from Each Segment
- **IBI (HRV)**: SDNN, RMSSD (2 features)
- **EDA**: mean, std, max (3 features)
- **HR**: mean, std, max (3 features)
- **TEMP**: mean, std (2 features)
- **ACC**: mean, std (2 features)

Feature Extraction: `extract_segment_features()`
- Converts raw time series samples → 16 numbers
- Mean HR = average cardiovascular load
- HRV RMSSD = autonomic nervous system flexibility
- EDA std = sweat response variability

In [ ]:
def extract_segment_features(subject, segment, signals, fs_dict):
    """Extract all biomarkers for one segment"""
    features = {
        'subject': subject,
        'label': segment['label'],
        'phase': segment['phase'],
        'duration': segment['end'] - segment['start']
    }
    
    start_t, end_t = segment['start'], segment['end']
    
    # HRV
    if 'IBI' in signals and len(signals['IBI']) > 0:
        hrv = calculate_hrv(signals['IBI'], start_t, end_t)
        features.update(hrv)
    else:
        features.update({'SDNN': np.nan, 'RMSSD': np.nan})
    
    # EDA
    if 'EDA' in signals:
        fs = fs_dict[subject]['EDA']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        eda = signals['EDA'][idx_start:idx_end].flatten()
        features['EDA_mean'] = np.mean(eda)
        features['EDA_std'] = np.std(eda)
        features['EDA_max'] = np.max(eda)
    
    # HR
    if 'HR' in signals:
        fs = fs_dict[subject]['HR']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        hr = signals['HR'][idx_start:idx_end].flatten()
        features['HR_mean'] = np.mean(hr)
        features['HR_std'] = np.std(hr)
        features['HR_max'] = np.max(hr)
    
    # Temperature
    if 'TEMP' in signals:
        fs = fs_dict[subject]['TEMP']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        temp = signals['TEMP'][idx_start:idx_end].flatten()
        features['TEMP_mean'] = np.mean(temp)
        features['TEMP_std'] = np.std(temp)
    
    # Accelerometer
    if 'ACC' in signals:
        fs = fs_dict[subject]['ACC']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        acc = signals['ACC'][idx_start:idx_end]
        acc_filtered = moving_average(acc)
        features['ACC_mean'] = np.mean(acc_filtered)
        features['ACC_std'] = np.std(acc_filtered)
    
    return features

print("Feature extraction function defined")

In [ ]:
# Extract features for all segments
all_features = []

for subject in subjects:
    if subject in all_segments:
        for segment in all_segments[subject]:
            feat = extract_segment_features(subject, segment, signal_data[subject], fs_dict)
            all_features.append(feat)

df_features = pd.DataFrame(all_features)

print(f"\n✓ Extracted {len(df_features)} segments")
print(f"   EXERCISE: {len(df_features[df_features['label']=='EXERCISE'])}")
print(f"   REST: {len(df_features[df_features['label']=='REST'])}")
print(f"\nColumns: {list(df_features.columns)}")
df_features

### Random Forest

In [ ]:
# Prepare data for Random Forest classifier
X = df_features.drop(columns=['subject', 'label', 'phase', 'duration'])
y = df_features['label']

# Handle missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42, stratify=y)

# Create and train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

### AdaBoost

In [ ]:
# compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = dict(zip(np.unique(y), class_weights))
#print(f"Class weights: {class_weight_dict}\n")

# train
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)
sample_weights = np.array([class_weight_dict[label] for label in y_train]) # for class imbalance
ada_model.fit(X_train, y_train, sample_weight=sample_weights)

# evaluate
y_pred_ada = ada_model.predict(X_test)
print("AdaBoost Classification Report:")
print(classification_report(y_test, y_pred_ada))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada))

#feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': ada_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 Important Features:")
print(feature_importance.head(10))

### XGBoost

In [ ]:
# encode labels to 0 & 1 (exercise -> 0, rest -> 1, bc LabelEncoder encodes alphabetically)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
print(f"Label encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
n_class_0 = len(y_train_encoded[y_train_encoded == 0])
n_class_1 = len(y_train_encoded[y_train_encoded == 1])
scale_pos_weight = n_class_0 / n_class_1
print(f"scale_pos_weight: {scale_pos_weight:.4f}")
print(f"Class 0 count (train set): {n_class_0}, Class 1 count (train set): {n_class_1}\n")

# train
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=scale_pos_weight, # for class imbalance
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train_encoded)

# evaluate
y_pred_xgb_encoded = xgb_model.predict(X_test)

# convert predictions back to original labels 
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

#feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 Important Features:")
print(feature_importance.head(10))

### Support Vector Classifier

In [ ]:
# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# train
svc_model = SVC(
    kernel='rbf',
    class_weight='balanced',
    random_state=42
)
svc_model.fit(X_train_scaled, y_train)

# evaluate
y_pred_svc = svc_model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred_svc))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svc))

#feature importance
perm_importance = permutation_importance(
    svc_model,
    X_test_scaled,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring='f1_macro'
)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': perm_importance.importances_mean
}).sort_values('importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))